#  User Analytics & Loyalty Bonus Assignment
This project analyzes user deposits, withdrawals, and gameplay to compute loyalty scores and distribute bonuses.

## User Analytics & Loyalty Bonus Assignment

This analysis focuses on understanding user behavior based on:
- Deposits
- Withdrawals
- Gameplay activity

The goal is to calculate loyalty points and distribute bonuses accordingly.

##  Data Loading
We load deposit, withdrawal, and gameplay datasets to analyze user behavior.

### Data Loading

We load three datasets:
- Deposit Data
- Withdrawal Data
- Gameplay Data

These datasets help analyze user financial and engagement behavior.

In [56]:
import pandas as pd
import numpy as np

In [57]:
file_path = "Analytics Position Case Study.xlsx"

deposit = pd.read_excel(file_path, sheet_name='Deposit Data', skiprows=3)
withdrawal = pd.read_excel(file_path, sheet_name='Withdrawal Data', skiprows=3)
gameplay = pd.read_excel(file_path, sheet_name='User Gameplay data', skiprows=3)

##  Data Inspection
We inspect datasets using `.head()` to understand structure and columns.

**Insight:** Data requires cleaning before analysis.

In [58]:
gameplay.head()
deposit.head()
withdrawal.head()

,User Id,Datetime,Amount
0,190,2022-01-10 00:03:00,5872
1,159,2022-01-10 00:16:00,9540
2,164,2022-01-10 00:24:00,815
3,946,2022-01-10 00:29:00,23000
4,763,2022-01-10 00:40:00,9473


##  Initial Data Inspection
We preview the datasets to:
- Understand structure
- Identify column names
- Check for inconsistencies

### Insight:
- Data contains user-level transactional and gameplay information
- Column cleaning will be required for consistency

In [59]:
deposit.columns = deposit.columns.str.strip().str.lower()
withdrawal.columns = withdrawal.columns.str.strip().str.lower()
gameplay.columns = gameplay.columns.str.strip().str.lower()

##  Data Cleaning
Column names are standardized for consistency.

**Insight:** Prevents errors in merging and analysis.

###  Data Cleaning

Column names are standardized by:
- Removing spaces
- Converting to lowercase

### Insight:
- Ensures consistency across datasets
- Prevents errors during merging and aggregation

In [60]:
print(deposit.columns)
print(withdrawal.columns)
print(gameplay.columns)

Index(['user id', 'datetime', 'amount'], dtype='object')
Index(['user id', 'datetime', 'amount'], dtype='object')
Index(['user id', 'games played', 'datetime'], dtype='object')


In [61]:
deposit_summary = deposit.groupby('user id')['amount'].sum().reset_index()
deposit_summary.rename(columns={'amount': 'total_deposit'}, inplace=True)

##  Deposit Analysis
Total deposits per user are calculated.

**Insight:** Identifies high-value users contributing revenue.

### Deposit Analysis

We calculate total deposit per user.

### Insight:
- Helps identify high-value users
- Indicates users contributing maximum revenue

In [62]:
withdrawal_summary = withdrawal.groupby('user id')['amount'].sum().reset_index()
withdrawal_summary.rename(columns={'amount': 'total_withdrawal'}, inplace=True)

### Withdrawal Analysis

We calculate total withdrawal per user.

### Insight:
- High withdrawals may indicate:
  - Profit-taking users
  - Potential churn risk

##  Withdrawal Analysis
Total withdrawals per user are calculated.

**Insight:** High withdrawals may indicate churn or profit-taking.

In [63]:
games_summary = gameplay.groupby('user id')['games played'].sum().reset_index()

### Gameplay Analysis

We calculate total games played per user.

### Insight:
- Higher gameplay = higher engagement
- Important for loyalty and retention strategies

In [64]:
df = deposit_summary.merge(withdrawal_summary, on='user id', how='outer')
df = df.merge(games_summary, on='user id', how='outer')

df.fillna(0, inplace=True)

## Gameplay Analysis
Total gameplay activity per user is calculated.

**Insight:** Higher gameplay = higher engagement.

### Data Merging

All datasets are merged into a single dataframe.

### Insight:
- Provides a unified user-level view
- Missing values are replaced with 0 for accurate calculations

##  Loyalty Points Calculation

Loyalty points are calculated using:
- Deposits
- Withdrawals
- Gameplay activity

### Insight:
- Combines financial + engagement behavior
- Helps reward valuable users fairly

In [65]:
df['extra'] = df['total_deposit'] - df['total_withdrawal']
df['extra'] = df['extra'].apply(lambda x: max(x, 0))

df['loyalty_points'] = (
    0.01 * df['total_deposit'] +
    0.005 * df['total_withdrawal'] +
    0.001 * df['extra'] +
    0.2 * df['games played']
)

##  Data Merging
Datasets are merged into a unified user-level dataset.

**Insight:** Provides complete user view.

### User Ranking

Users are ranked based on loyalty points.

### Insight:
- Identifies top-performing users
- Useful for reward distribution and targeting

In [66]:
df = df.sort_values(by='loyalty_points', ascending=False)
df['rank'] = df['loyalty_points'].rank(ascending=False)

###  Bonus Distribution

Total bonus pool is distributed proportionally based on loyalty points.

### Insight:
- Ensures fair reward system
- Higher contribution → higher reward

##  Loyalty Points Calculation
Loyalty score is based on deposits, withdrawals, and gameplay.

**Insight:** Combines financial + engagement behavior.

In [67]:
total_bonus = 100000

df['bonus'] = (df['loyalty_points'] / df['loyalty_points'].sum()) * total_bonus

In [68]:
df.head()

,user id,total_deposit,total_withdrawal,games played,extra,loyalty_points,rank,bonus
634,634,515000.0,15737705.0,24,0.0,83843.325,1.0,5561.660654
672,672,2158700.0,233750.0,10,1924950.0,24682.700,2.0,1637.301496
99,99,1164800.0,2403141.0,10,0.0,23665.705,3.0,1569.840179
212,212,1924981.0,589850.0,1,1335131.0,23534.391,4.0,1561.129600
566,566,1819175.0,185071.0,183,1634104.0,20787.809,5.0,1378.937911


##  Slot-wise Loyalty Points Calculation
We calculate loyalty points for specific dates and time slots:
- S1: 12 AM to 12 PM
- S2: 12 PM to 12 AM

In [69]:
# Clean column names 
gameplay.columns = gameplay.columns.str.lower().str.replace(" ", "_")
deposit.columns = deposit.columns.str.lower().str.replace(" ", "_")

print(gameplay.columns)
print(deposit.columns)

gameplay['loyalty_points'] = gameplay['games_played'] * 10

Index(['user_id', 'games_played', 'datetime'], dtype='object')
Index(['user_id', 'datetime', 'amount'], dtype='object')


In [70]:
# Convert datetime
gameplay['datetime'] = pd.to_datetime(gameplay['datetime'])

# Create date & hour
gameplay['date'] = gameplay['datetime'].dt.date
gameplay['hour'] = gameplay['datetime'].dt.hour

# Create slot column
def get_slot(hour):
    if 0 <= hour < 12:
        return 'S1'
    else:
        return 'S2'

gameplay['slot'] = gameplay['hour'].apply(get_slot)

# Function for slot-wise loyalty
def calculate_slot_loyalty(date, slot):
    df = gameplay[
        (gameplay['date'] == pd.to_datetime(date).date()) &
        (gameplay['slot'] == slot)
    ]
    
    result = df.groupby('user_id')['loyalty_points'].sum().reset_index()
    return result

# Required outputs
slot_1 = calculate_slot_loyalty('2022-10-02', 'S1')
slot_2 = calculate_slot_loyalty('2022-10-16', 'S2')
slot_3 = calculate_slot_loyalty('2022-10-18', 'S1')
slot_4 = calculate_slot_loyalty('2022-10-26', 'S2')

print("2nd Oct S1:\n", slot_1)
print("\n16th Oct S2:\n", slot_2)
print("\n18th Oct S1:\n", slot_3)
print("\n26th Oct S2:\n", slot_4)

2nd Oct S1:
 Empty DataFrame
Columns: [user_id, loyalty_points]
Index: []

16th Oct S2:
      user_id  loyalty_points
0          2              20
1          5              90
2          6              10
3          8              50
4          9             510
..       ...             ...
510      990             620
511      991              10
512      992             440
513      996              90
514      999              10

[515 rows x 2 columns]

18th Oct S1:
      user_id  loyalty_points
0          2              20
1          3              20
2          5              80
3          7              30
4          8              90
..       ...             ...
542      990             720
543      992             470
544      996              50
545      997              10
546      999              20

[547 rows x 2 columns]

26th Oct S2:
      user_id  loyalty_points
0          5              40
1          6              10
2          7              10
3          8         

##  Average Metrics

In [71]:
# Average deposit amount
avg_deposit = deposit['amount'].mean()
print("Average Deposit Amount:", avg_deposit)

# Average deposit per user
avg_deposit_user = deposit.groupby('user_id')['amount'].sum().mean()
print("Average Deposit per User:", avg_deposit_user)

# Average games per user
avg_games = gameplay.groupby('user_id').size().mean()
print("Average Games per User:", avg_games)

Average Deposit Amount: 5492.185399701801
Average Deposit per User: 104669.64918032786
Average Games per User: 355.266


In [72]:
# Clean column names (if not already done)
withdrawal.columns = withdrawal.columns.str.strip().str.lower()

# Check columns
print(withdrawal.columns)

Index(['user id', 'datetime', 'amount'], dtype='object')


##  User Ranking
Users ranked based on loyalty points.

**Insight:** Top users are most valuable.

In [73]:
withdrawal_summary = withdrawal.groupby('user id')['amount'].sum().reset_index()
withdrawal_summary.rename(columns={'amount': 'total_withdrawal'}, inplace=True)

withdrawal_summary.head()


,user id,total_withdrawal
0,2,1270215
1,5,32700
2,7,6617
3,9,171456
4,11,101500


In [74]:
gameplay.columns = gameplay.columns.str.strip().str.lower()
print(gameplay.columns)

Index(['user_id', 'games_played', 'datetime', 'loyalty_points', 'date', 'hour',
       'slot'],
      dtype='object')


In [75]:
print(gameplay.columns)

Index(['user_id', 'games_played', 'datetime', 'loyalty_points', 'date', 'hour',
       'slot'],
      dtype='object')


##  Bonus Distribution
Bonus distributed proportionally based on loyalty points.

**Insight:** Fair reward system.

In [76]:
gameplay_summary = gameplay.groupby('user_id')['games_played'].sum().reset_index()
gameplay_summary.rename(columns={'games_played': 'total_games_played'}, inplace=True)

gameplay_summary.head()

,user_id,total_games_played
0,0,15
1,1,8
2,2,97
3,3,80
4,4,5


##  Bonus Distribution Strategy

The bonus pool of Rs 50,000 is distributed among the top 50 players based on their loyalty points.

### Why Loyalty Points?
- Loyalty points reflect both **deposit behavior** and **game activity**
- It ensures players contributing more to the platform get higher rewards

### Alternative Approaches Considered:
- Based only on number of games → ignores monetary contribution
- Based only on deposits → ignores engagement

### Recommended Approach:
A **hybrid model** (loyalty points) is the best as it balances:
- Engagement (games played)
- Revenue contribution (deposits)

Thus, proportional distribution using loyalty points is fair and effective.

In [77]:
deposit_summary.columns = deposit_summary.columns.str.lower().str.replace(" ", "_")
withdrawal_summary.columns = withdrawal_summary.columns.str.lower().str.replace(" ", "_")
gameplay_summary.columns = gameplay_summary.columns.str.lower().str.replace(" ", "_")
final_df = deposit_summary.merge(withdrawal_summary, on='user_id', how='outer')
final_df = final_df.merge(gameplay_summary, on='user_id', how='outer')

final_df.head()

,user_id,total_deposit,total_withdrawal,total_games_played
0,0,NaN,NaN,15
1,1,5000.0,NaN,8
2,2,567000.0,1270215.0,97
3,3,40000.0,NaN,80
4,4,1750.0,NaN,5


In [78]:
final_df.fillna(0, inplace=True)

##  Key Insights
- Few users contribute most deposits
- Gameplay ≠ always high spending
- Some users show churn risk
- Loyalty scoring helps segmentation

In [79]:
final_df['net_balance'] = final_df['total_deposit'] - final_df['total_withdrawal']
final_df['avg_deposit'] = final_df['total_deposit']  # can refine later

final_df.head()

,user_id,total_deposit,total_withdrawal,total_games_played,net_balance,avg_deposit
0,0,0.0,0.0,15,0.0,0.0
1,1,5000.0,0.0,8,5000.0,5000.0
2,2,567000.0,1270215.0,97,-703215.0,567000.0
3,3,40000.0,0.0,80,40000.0,40000.0
4,4,1750.0,0.0,5,1750.0,1750.0


In [80]:
top_users = final_df.sort_values(by='total_deposit', ascending=False).head(10)
top_users

,user_id,total_deposit,total_withdrawal,total_games_played,net_balance,avg_deposit
672,672,2158700.0,233750.0,10,1924950.0,2158700.0
212,212,1924981.0,589850.0,1,1335131.0,1924981.0
566,566,1819175.0,185071.0,183,1634104.0,1819175.0
740,740,1738490.0,365288.0,2,1373202.0,1738490.0
714,714,1676300.0,0.0,6,1676300.0,1676300.0
30,30,1329000.0,152145.0,13,1176855.0,1329000.0
222,222,1285000.0,99358.0,10,1185642.0,1285000.0
569,569,1227780.0,0.0,38,1227780.0,1227780.0
99,99,1164800.0,2403141.0,10,-1238341.0,1164800.0
352,352,1083034.0,429518.0,313,653516.0,1083034.0


##  Conclusion
- Retain high-value users
- Convert engaged users to paying users
- Monitor high-withdrawal users
- Use loyalty-based rewards

In [81]:
final_df.head()

,user_id,total_deposit,total_withdrawal,total_games_played,net_balance,avg_deposit
0,0,0.0,0.0,15,0.0,0.0
1,1,5000.0,0.0,8,5000.0,5000.0
2,2,567000.0,1270215.0,97,-703215.0,567000.0
3,3,40000.0,0.0,80,40000.0,40000.0
4,4,1750.0,0.0,5,1750.0,1750.0


In [82]:
final_df['net_balance'] = final_df['total_deposit'] - final_df['total_withdrawal']
final_df['avg_games_per_user'] = final_df['total_games_played']

final_df.head()

,user_id,total_deposit,total_withdrawal,total_games_played,net_balance,avg_deposit,avg_games_per_user
0,0,0.0,0.0,15,0.0,0.0,15
1,1,5000.0,0.0,8,5000.0,5000.0,8
2,2,567000.0,1270215.0,97,-703215.0,567000.0,97
3,3,40000.0,0.0,80,40000.0,40000.0,80
4,4,1750.0,0.0,5,1750.0,1750.0,5


In [83]:
top_users = final_df.sort_values(by='total_deposit', ascending=False).head(10)
top_users

,user_id,total_deposit,total_withdrawal,total_games_played,net_balance,avg_deposit,avg_games_per_user
672,672,2158700.0,233750.0,10,1924950.0,2158700.0,10
212,212,1924981.0,589850.0,1,1335131.0,1924981.0,1
566,566,1819175.0,185071.0,183,1634104.0,1819175.0,183
740,740,1738490.0,365288.0,2,1373202.0,1738490.0,2
714,714,1676300.0,0.0,6,1676300.0,1676300.0,6
30,30,1329000.0,152145.0,13,1176855.0,1329000.0,13
222,222,1285000.0,99358.0,10,1185642.0,1285000.0,10
569,569,1227780.0,0.0,38,1227780.0,1227780.0,38
99,99,1164800.0,2403141.0,10,-1238341.0,1164800.0,10
352,352,1083034.0,429518.0,313,653516.0,1083034.0,313


Insights

* Users with highest deposits are key revenue drivers
* Some users withdraw more than they deposit → potential churn
* High gameplay users indicate strong engagement
* Few users dominate activity (Pareto pattern likely)

Conclusion:

* Focus on retaining high-deposit users
* Encourage low-deposit but high-gameplay users to convert
* Monitor users with high withdrawals

##  Part C: Evaluation of Loyalty Point Formula

### Is the current loyalty formula fair?

The current loyalty formula is partially fair because:
- It rewards both **player activity** and **financial contribution**
- Encourages engagement on the platform

### Limitations:
- High deposit users may dominate rankings unfairly
- Does not consider **frequency vs quality of gameplay**
- No penalty for withdrawals or inactive behavior

### Suggestions to Improve:

1. **Normalize Deposits**
   - Prevent very high deposit users from dominating

2. **Introduce Recency Factor**
   - Give more weight to recent activity

3. **Gameplay Quality Weight**
   - Reward meaningful gameplay instead of just quantity

4. **Penalty for High Withdrawals**
   - Reduce points for users withdrawing too frequently

5. **Capping System**
   - Limit maximum points from a single activity

### Conclusion:
The formula is a good starting point but can be made more robust by balancing fairness, engagement, and revenue contribution.